# 02 — Le retrieval hybride, étape par étape

Le cœur technique du projet : on ne cherche pas avec UNE méthode mais **trois,
complémentaires**, qu'on fusionne puis qu'on affine :

1. **Sémantique** (embeddings bge-m3) — capte le *sens* (« CGM » ↔ « centre de gestion »).
2. **BM25** (mots-clés) — capte les *termes exacts* (identifiants, acronymes, codes).
3. **GraphRAG** (graphe d'entités) — capte les *relations* (désactivé par défaut, cf. A/B).

Puis **fusion RRF** (Reciprocal Rank Fusion) → **reranking cross-encoder** (le plus précis,
mais coûteux, donc seulement sur le pool fusionné).

> Prérequis : Ollama + MongoDB démarrés, corpus ingéré. ~10-20 s d'exécution.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from core.ask import _get_vector_store, _load_bm25
from retrieval.semantic_search import run_semantic_for_query
from retrieval.keyword_bm25 import run_bm25_for_query
from retrieval.retrieve import hybrid_retrieve

def show(lookup, ids, n=3):
    for r, cid in enumerate(ids[:n], 1):
        p = lookup.get(cid, {})
        src = p.get('meta', {}).get('source', '?')
        snippet = (p.get('doc', '') or '').replace(chr(10), ' ')[:80]
        print(f'  {r}. [{src}] {snippet}')

QUESTION = "Que d\u00e9signe l'acronyme CGM dans le syst\u00e8me Mistral ?"
print('Question :', QUESTION)

C:\Users\laury\Desktop\rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Question : Que désigne l'acronyme CGM dans le système Mistral ?


## Leg 1 — recherche sémantique (sens)

In [2]:
store = _get_vector_store()
sem_ids, sem_lookup = run_semantic_for_query(store, QUESTION, topn=5)
print(f'{len(sem_ids)} chunks (sémantique) — top 3 :')
show(sem_lookup, sem_ids)

5 chunks (sémantique) — top 3 :
  1. [ANSSI-CC-cible_2011-1-20.md] ## Connexion entre le poste CGM et le boîtier mistral Frontal  Le poste CGM est 
  2. [ANSSI-CC-cible_2011-1-20.md] [2. Présentation de la cible d'évaluation (TOE)] ## 2.1  Le système MISTRAL VPN 
  3. [ANSSI-CC-cible_2011-1-20.md] ## Redondance intra-site et inter-site  La  redondance  de  l'administration  Mi


## Leg 2 — recherche BM25 (mots-clés exacts)

In [3]:
bm25_tuple = _load_bm25(None)
if bm25_tuple:
    bm_ids, bm_lookup = run_bm25_for_query(bm25_tuple, QUESTION, topn=5)
    print(f'{len(bm_ids)} chunks (BM25) — top 3 :')
    show(bm_lookup, bm_ids)
    only_bm25 = [i for i in bm_ids if i not in sem_ids]
    print(f'\n=> {len(only_bm25)} chunk(s) trouvé(s) par BM25 mais PAS par la sémantique (complémentarité).')
else:
    print('Pas d\'index BM25 (ingestion requise).')

5 chunks (BM25) — top 3 :
  1. [ANSSI-CC-cible_2011-1-20.md] [2. Présentation de la cible d'évaluation (TOE) > 2.3 Identification de la TOE e
  2. [ANSSI-CC-cible_2011-1-20.md] [Description] Ce tableau semble être un résumé des hypothèses, menaces et object
  3. [ANSSI-CC-cible_2011-1-20.md] - -la  redondance différenciée : l'ensemble du système  Mistral a la capacité d'

=> 4 chunk(s) trouvé(s) par BM25 mais PAS par la sémantique (complémentarité).


## Fusion RRF + reranking cross-encoder

`hybrid_retrieve` fusionne les listes (RRF, pondéré sémantique/BM25), puis **réordonne**
le pool avec le cross-encoder `bge-reranker-v2-m3` (qui lit *vraiment* la paire question↔chunk).
Le score CE sert aussi au **filtre hors-scope** (sous un seuil → « pas dans le corpus »).

In [4]:
final_chunks, max_ce = hybrid_retrieve(
    collection=store, query=QUESTION, bm25_tuple=bm25_tuple, rerank_on=True, debug=False,
)
print(f'Classement FINAL (fusion + rerank) — meilleur score CE = {max_ce:.3f}\n')
for r, c in enumerate(final_chunks[:5], 1):
    ce = c.get('ce_score')
    src = c.get('meta', {}).get('source', '?')
    snippet = (c.get('doc', '') or '').replace(chr(10), ' ')[:80]
    ce_str = f'{ce:.3f}' if isinstance(ce, (int, float)) else ' - '
    print(f'  {r}. CE={ce_str}  [{src}] {snippet}')

21:06:57 INFO    rag.rerank | [rerank] Chargement Cross-Encoder local: C:\Users\laury\Desktop\rag_project\models\bge-reranker-v2-m3 (device=cuda:0)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8931.26it/s]

Classement FINAL (fusion + rerank) — meilleur score CE = 0.714

  1. CE=0.714  [ANSSI-CC-cible_2011-1-20.md] [2. Présentation de la cible d'évaluation (TOE) > 2.1 Le système MISTRAL VPN IP]
  2. CE=0.712  [ANSSI-CC-cible_2011-1-20.md] [2. Présentation de la cible d'évaluation (TOE)] ## 2.1  Le système MISTRAL VPN 
  3. CE=0.609  [ANSSI-CC-cible_2011-1-20.md] - Le  Centre  de  Gestion  centralise  la  description  du  réseau  du  client, 
  4. CE=0.568  [ANSSI-CC-cible_2011-1-20.md] ## Redondance intra-site et inter-site  La  redondance  de  l'administration  Mi
  5. CE=0.555  [ANSSI-CC-cible_2011-1-20.md] [2. Présentation de la cible d'évaluation (TOE) > 2.3 Identification de la TOE e


## À retenir

- **Sémantique et BM25 ne ramènent pas les mêmes chunks** → les fusionner couvre à la fois
  le *sens* et les *termes exacts* (crucial pour des docs techniques pleins d'identifiants).
- Le **cross-encoder réordonne** le pool fusionné : c'est lui qui donne le classement final
  (et le signal *hors-scope*).
- Tout est **mesuré** : la qualité de ce retrieval est chiffrée dans [`01_evaluation.ipynb`](01_evaluation.ipynb)
  (hit@k 0.93, recall 0.90). GraphRAG, lui, a été mesuré **neutre** sur le retrieval factuel
  → désactivé par défaut (décision A/B, pas un oubli).